In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import time

# --- Model loading ---
import joblib

# --- Evaluation metrics ---
from sklearn.metrics import (
    average_precision_score,    # PR-AUC — our headline metric
    precision_recall_curve,     # returns (precision, recall, thresholds) arrays
    roc_auc_score,              # ROC-AUC — secondary metric
    roc_curve,                  # returns (fpr, tpr, thresholds) for plotting
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    classification_report,
)

warnings.filterwarnings("ignore")

# ── Reproducibility ──────────────────────────────────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ── Plot style ───────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
FRAUD_PALETTE = {0: "#2196F3", 1: "#F44336"}   # Blue = legit, Red = fraud

# ── File paths ───────────────────────────────────────────────────────────────
PARQUET_TEST  = os.path.join("data", "test.parquet")
MODEL_LR      = os.path.join("models", "lr_baseline.joblib")
MODEL_XGB_SPW = os.path.join("models", "xgb_spw.joblib")
MODEL_XGB_SMO = os.path.join("models", "xgb_smote.joblib")
MODEL_META    = os.path.join("models", "model_meta.joblib")
FIGURES_DIR   = os.path.join("reports", "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

FN_COST = 500   # $ per missed fraud (false negative)
FP_COST = 10    # $ per wrongly flagged legitimate transaction (false positive)

print(" Imports and configuration complete.")
print(f"   Cost assumption: ${FN_COST}/FN  ·  ${FP_COST}/FP  "
      f"→  ratio {FN_COST // FP_COST}:1  (model should lean toward recall)")

## Model Evaluation

In [ ]:
for path in [PARQUET_TEST, MODEL_LR, MODEL_XGB_SPW, MODEL_XGB_SMO, MODEL_META]:
    assert os.path.exists(path), (
        f"\n  Missing: {path}\n"
        f"    → Run all cells in fraud_detection_stage7_8.ipynb first."
    )

# ── Load metadata ────────────────────────────────────────────────────────────
# model_meta carries the exact FEATURE_COLS list in training order.
# Using this list — not re-deriving it — guarantees the test features are
# assembled in the identical column order the model was fitted on.
# Column order is critical for tree models: position 0, 1, 2 ... are what
# the model uses internally; names are ignored at predict time.
meta         = joblib.load(MODEL_META)
FEATURE_COLS   = meta['feature_cols']

# ── Load test set ────────────────────────────────────────────────────────────
# test.parquet contains both feature columns AND the isFraud label.
# We slice using the exact FEATURE_COLS order established in Stage 6.
test   = pd.read_parquet(PARQUET_TEST)
X_test = test[FEATURE_COLS]
y_test = test["isFraud"].astype(int)

# ── Load the three trained models ────────────────────────────────────────────
pipe_lr        = joblib.load(MODEL_LR)       # Logistic Regression pipeline
xgb_spw        = joblib.load(MODEL_XGB_SPW)  # XGBoost with scale_pos_weight
pipe_xgb_smote = joblib.load(MODEL_XGB_SMO)  # XGBoost + SMOTE pipeline

print(f"Test set:        {X_test.shape[0]:>9,} rows × {X_test.shape[1]} features")
print(f"Fraud in test:   {y_test.sum():>9,} ({y_test.mean()*100:.4f}%)")
print()
print("Models loaded:")
print("  ✓ Logistic Regression baseline   (pipe_lr)")
print("  ✓ XGBoost scale_pos_weight       (xgb_spw)   ← primary candidate")
print("  ✓ XGBoost + SMOTE                (pipe_xgb_smote)")
print()
print(f"Feature columns ({len(FEATURE_COLS)} total): {FEATURE_COLS}")

In [ ]:
# Before running a single prediction, it's worth being explicit about which
# numbers matter and in what order. This shapes how you read every result below.
#
# ─────────────────────────────────────────────────────────────────────────────
# METRIC 1: PR-AUC (Average Precision)           
# ─────────────────────────────────────────────────────────────────────────────
# The area under the Precision-Recall curve across all decision thresholds.
# It looks ONLY at the positive (fraud) class — true positives, false positives,
# and false negatives. The vast majority of true negatives play NO role. So it
# cannot be inflated by the model correctly ignoring millions of obvious legit
# transactions. Its baseline is the fraud rate itself (~0.0013), so any real
# skill stands out dramatically against that near-zero floor.
#
# ─────────────────────────────────────────────────────────────────────────────
# METRIC 2: ROC-AUC                              
# ─────────────────────────────────────────────────────────────────────────────
# Area under the ROC curve (TPR vs FPR). The FPR denominator includes the
# huge true-negative count, so even many FPs in absolute terms barely move it.
# Result: ROC-AUC looks optimistically high at extreme imbalance. Still worth
# reporting for comparability with published work, but don't lead with it.
#
# ─────────────────────────────────────────────────────────────────────────────
# METRIC 3: F1 / Precision / Recall at a threshold  
# ─────────────────────────────────────────────────────────────────────────────
# These are per-threshold numbers — they convert the model's probability output
# into a binary decision and measure how good that decision is.
# F1 (harmonic mean of precision and recall) is only high when BOTH are decent.
# The arithmetic mean would let one strong number mask a terrible one; the
# harmonic mean refuses to.
#
# ─────────────────────────────────────────────────────────────────────────────
# METRIC 4: Confusion Matrix                     
# ─────────────────────────────────────────────────────────────────────────────
# TP/FP/FN/TN counts that translate directly into dollars:
# "At this threshold we miss 12 frauds per day, costing $6,000 in losses."
#
# ─────────────────────────────────────────────────────────────────────────────
# ✗ ACCURACY                                     
# ─────────────────────────────────────────────────────────────────────────────
# A "predict always legitimate" model scores 99.87% accuracy while catching
# zero fraud. Accuracy is dominated by the majority class and gives a completely
# false picture. We show it in Cell 10 once — to make this point explicit.

print("Metric hierarchy (no code in this cell — read the comments above).")
print("  1. PR-AUC   ← headline (threshold-independent, fraud-class only)")
print("  2. ROC-AUC  ← secondary (report for comparability)")
print("  3. F1 / Precision / Recall  ← at a chosen threshold")
print("  4. Confusion matrix  ← translates to dollar impact")
print("  ✗ Accuracy  ← never use on imbalanced fraud data")

In [ ]:
print("⏳ Generating probability scores on the test set...")
t0 = time.time()

y_prob_lr    = pipe_lr.predict_proba(X_test)[:, 1]
y_prob_spw   = xgb_spw.predict_proba(X_test)[:, 1]
y_prob_smote = pipe_xgb_smote.predict_proba(X_test)[:, 1]

print(f" Predictions complete in {time.time() - t0:.2f}s")
print()
print("Predicted fraud probability stats (test set):")
for name, probs in [("LR Baseline", y_prob_lr),
                     ("XGB (SPW)  ", y_prob_spw),
                     ("XGB (SMOTE)", y_prob_smote)]:
    print(f"  {name}:  mean={probs.mean():.4f}  "
          f"median={np.median(probs):.4f}  "
          f"p95={np.percentile(probs, 95):.4f}  "
          f"max={probs.max():.4f}")

# Dicts for clean iteration in all downstream cells
model_probs = {
    "LR Baseline" : y_prob_lr,
    "XGB (SPW)"   : y_prob_spw,
    "XGB (SMOTE)" : y_prob_smote,
}
MODEL_COLORS = {
    "LR Baseline" : "#9C27B0",   # Purple
    "XGB (SPW)"   : "#FF5722",   # Deep orange  (primary model)
    "XGB (SMOTE)" : "#009688",   # Teal
}

In [ ]:
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print("  PR-AUC (Average Precision) — Primary Metric")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

pr_auc_results = {}
for name, probs in model_probs.items():
    ap = average_precision_score(y_test, probs)
    pr_auc_results[name] = ap
    print(f"  {name:<20}: PR-AUC = {ap:.4f}")

fraud_rate      = y_test.mean()
best_model_name = max(pr_auc_results, key=pr_auc_results.get)
best_pr_auc     = pr_auc_results[best_model_name]

print(f"\n  Random baseline         : PR-AUC = {fraud_rate:.4f}  (= fraud rate)")
print(f"\n   Best model: {best_model_name}  (PR-AUC = {best_pr_auc:.4f})")
print(f"     Lift over random baseline: {best_pr_auc / fraud_rate:.0f}×")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

for name, probs in model_probs.items():
    precision_arr, recall_arr, _ = precision_recall_curve(y_test, probs)
    ap = pr_auc_results[name]
    ax.plot(recall_arr, precision_arr,
            color=MODEL_COLORS[name],
            linewidth=2.0,
            label=f"{name}  (PR-AUC = {ap:.4f})")

ax.axhline(y=fraud_rate, color="grey", linestyle="--", linewidth=1.2,
           label=f"No-skill baseline  (PR-AUC ≈ {fraud_rate:.4f})")
ax.set_xlabel("Recall  (fraction of real frauds caught)", fontsize=12)
ax.set_ylabel("Precision  (fraction of flags that are real fraud)", fontsize=12)
ax.set_title("Precision-Recall Curves — All Three Models", fontsize=14, fontweight="bold")
ax.legend(loc="upper right", fontsize=10)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])
plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "pr_curves_comparison.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print(f" Figure saved → {FIGURES_DIR}/pr_curves_comparison.png")

In [ ]:
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print("  ROC-AUC — Secondary Metric")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

roc_auc_results = {}
for name, probs in model_probs.items():
    ra = roc_auc_score(y_test, probs)
    roc_auc_results[name] = ra
    print(f"  {name:<20}: ROC-AUC = {ra:.4f}")

print(f"\n  Random baseline         : ROC-AUC = 0.5000")

# ── ROC curves plot ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))
for name, probs in model_probs.items():
    fpr_arr, tpr_arr, _ = roc_curve(y_test, probs)
    ra = roc_auc_results[name]
    ax.plot(fpr_arr, tpr_arr, color=MODEL_COLORS[name], linewidth=2,
            label=f"{name}  (ROC-AUC = {ra:.4f})")

ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random baseline  (0.5)")
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate (Recall)", fontsize=12)
ax.set_title("ROC Curves — All Three Models", fontsize=14, fontweight="bold")
ax.legend(loc="lower right", fontsize=10)
plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "roc_curves_comparison.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print(f" Figure saved → {FIGURES_DIR}/roc_curves_comparison.png")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, probs) in zip(axes, model_probs.items()):
    y_pred_d = (probs >= 0.5).astype(int)
    cm_d = confusion_matrix(y_test, y_pred_d, labels=[0, 1])
    ConfusionMatrixDisplay(cm_d, display_labels=["Legitimate", "Fraud"]).plot(
        ax=ax, colorbar=False, cmap="Blues"
    )
    tn_d, fp_d, fn_d, tp_d = cm_d.ravel()
    rec_d  = tp_d / (tp_d + fn_d + 1e-9)
    prec_d = tp_d / (tp_d + fp_d + 1e-9)
    f1_d   = f1_score(y_test, y_pred_d, zero_division=0)
    ax.set_title(
        f"{name}\nRecall={rec_d:.3f}  Precision={prec_d:.3f}  F1={f1_d:.3f}",
        fontsize=10, fontweight="bold"
    )

plt.suptitle("Confusion Matrices at Default Threshold (0.5)", fontsize=13,
             fontweight="bold", y=1.02)
plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "confusion_matrices_default_threshold.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print(f" Figure saved → {FIGURES_DIR}/confusion_matrices_default_threshold.png")

In [ ]:
THRESHOLD_DEFAULT = 0.5

for name, probs in model_probs.items():
    y_pred_d = (probs >= THRESHOLD_DEFAULT).astype(int)
    print(f"{'═' * 58}")
    print(f"  {name}   (threshold = {THRESHOLD_DEFAULT})")
    print(f"{'═' * 58}")
    print(classification_report(
        y_test, y_pred_d,
        target_names=["Legitimate (0)", "Fraud (1)"],
        digits=4
    ))

In [ ]:
y_never_fraud = np.zeros(len(y_test), dtype=int)

n_fraud = y_test.sum()
n_legit = (y_test == 0).sum()
n_total = len(y_test)

never_accuracy = n_legit / n_total * 100   # % of rows correctly labelled "legit"
never_recall   = 0.0                       # catches no fraud at all
never_f1       = 0.0                       # no TP → F1 = 0

print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print("  The Accuracy Paradox — A 'Never Fraud' Classifier")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(f"  Test set: {n_total:,} transactions  "
      f"({n_fraud:,} fraud  |  {n_legit:,} legitimate)")
print()
print(f"  'Never fraud' accuracy : {never_accuracy:.3f}%  ← looks impressive!")
print(f"  'Never fraud' recall   : {never_recall:.3f}     ← catches zero fraud")
print(f"  'Never fraud' F1       : {never_f1:.3f}     ← completely useless")
print()

# Compare against our best XGBoost model at 0.5 (note: threshold will improve in Stage 10)
y_pred_xgb_d = (y_prob_spw >= 0.5).astype(int)
xgb_accuracy = (y_pred_xgb_d == y_test).mean() * 100
xgb_recall   = (y_pred_xgb_d[y_test == 1] == 1).mean()
xgb_f1       = f1_score(y_test, y_pred_xgb_d, zero_division=0)

print(f"  XGBoost (SPW, threshold=0.5):")
print(f"    Accuracy : {xgb_accuracy:.3f}%  ← barely changed from 'never fraud'")
print(f"    Recall   : {xgb_recall:.3f}     ← actually catches real fraud")
print(f"    F1       : {xgb_f1:.3f}")
print()
print("     The accuracy numbers are almost identical.")
print("     But recall went from 0 to meaningful.")
print("     This is why accuracy is not just suboptimal — it is actively")
print("     misleading as a primary metric for imbalanced fraud detection.")

In [ ]:
rows = []
for name, probs in model_probs.items():
    y_pred_d = (probs >= 0.5).astype(int)
    tn_d, fp_d, fn_d, tp_d = confusion_matrix(y_test, y_pred_d, labels=[0, 1]).ravel()
    rec_d  = tp_d / (tp_d + fn_d + 1e-9)
    prec_d = tp_d / (tp_d + fp_d + 1e-9)
    rows.append({
        "Model"              : name,
        "PR-AUC"             : round(pr_auc_results[name], 4),
        "ROC-AUC"            : round(roc_auc_results[name], 4),
        "Recall @ 0.5"       : round(rec_d, 4),
        "Precision @ 0.5"    : round(prec_d, 4),
        "F1 @ 0.5"           : round(f1_score(y_test, y_pred_d, zero_division=0), 4),
        "TP (caught)"        : int(tp_d),
        "FN (missed)"        : int(fn_d),
        "FP (false alarms)"  : int(fp_d),
    })

summary_df = pd.DataFrame(rows).set_index("Model")

print("Model Comparison — All Metrics at Default Threshold (0.5):\n")
print(summary_df.to_string())
print()
print("Sort key: PR-AUC  (threshold-independent; the honest headline)")

In [ ]:
BEST_MODEL_NAME  = max(pr_auc_results, key=pr_auc_results.get)
BEST_MODEL_PROBS = model_probs[BEST_MODEL_NAME]
BEST_MODEL_OBJ   = {
    "LR Baseline" : pipe_lr,
    "XGB (SPW)"   : xgb_spw,
    "XGB (SMOTE)" : pipe_xgb_smote,
}[BEST_MODEL_NAME]

print(f"   Best model selected:  {BEST_MODEL_NAME}")
print(f"   PR-AUC  : {pr_auc_results[BEST_MODEL_NAME]:.4f}")
print(f"   ROC-AUC : {roc_auc_results[BEST_MODEL_NAME]:.4f}")
print()
print("   This model carries into:")
print("   → Stage 10 : decision threshold tuning")
print("   → Stage 11 : SHAP feature importance and explainability")
print("   → Stage 13 : joblib artifact bundle for the Streamlit app")

In [ ]:
print("=" * 62)
print("  STAGE 9 COMPLETE — EVALUATION SUMMARY")
print("=" * 62)
print()
for name in model_probs:
    print(f"  {name:<20}  "
          f"PR-AUC = {pr_auc_results[name]:.4f}   "
          f"ROC-AUC = {roc_auc_results[name]:.4f}")
print()
print(f"  Best model: {BEST_MODEL_NAME}  "
      f"(PR-AUC = {pr_auc_results[BEST_MODEL_NAME]:.4f})")
print()
print("  Key takeaways:")
print("    • PR-AUC is the right headline metric")
print("    • ROC-AUC is optimistic here; report but don't lead with it")
print("    • Accuracy is misleading — we demonstrated the paradox numerically")
print("    • Default 0.5 threshold is almost certainly not optimal (Stage 10)")
print()
print("  Figures saved to reports/figures/:")
for fname in ["pr_curves_comparison.png", "roc_curves_comparison.png",
              "confusion_matrices_default_threshold.png"]:
    print(f"    {fname}")
print()
print("  NEXT: Stage 10 — Decision Threshold Tuning")
print("  (find the threshold that minimises business cost, not just F1)")
print("=" * 62)

## Decision Threshold Tuning

In [ ]:
theoretical_cost_threshold = FP_COST / (FP_COST + FN_COST)

print("Why 0.5 is wrong for fraud detection:")
print()
print(f"  FN_COST           = ${FN_COST:,}  (per missed fraud)")
print(f"  FP_COST           = ${FP_COST:,}    (per wrongly flagged transaction)")
print(f"  Cost ratio FN:FP  = {FN_COST // FP_COST}:1")
print()
print(f"  Theoretical cost-optimal threshold (perfect calibration):")
print(f"    threshold* = {FP_COST} / ({FP_COST} + {FN_COST}) = {theoretical_cost_threshold:.4f}")
print()
print(f"  → We should be flagging transactions with P(fraud) as low as ~{theoretical_cost_threshold:.2f}")
print(f"  → The empirical optimum (Cell 17) will be in this neighbourhood")
print()
print("  The threshold is a BUSINESS DIAL, not a statistical constant.")
print("  Stage 10 turns it into a deliberate decision backed by dollar logic.")

In [ ]:
thresholds = np.linspace(0.01, 0.99, 200)

precisions = []
recalls    = []
f1s        = []
fn_counts  = []
fp_counts  = []

for t in thresholds:
    y_pred_t = (BEST_MODEL_PROBS >= t).astype(int)
    tn_t, fp_t, fn_t, tp_t = confusion_matrix(
        y_test, y_pred_t, labels=[0, 1]
    ).ravel()

    prec = tp_t / (tp_t + fp_t + 1e-9)
    rec  = tp_t / (tp_t + fn_t + 1e-9)
    f1   = 2 * prec * rec / (prec + rec + 1e-9)

    precisions.append(prec)
    recalls.append(rec)
    f1s.append(f1)
    fn_counts.append(int(fn_t))
    fp_counts.append(int(fp_t))

precisions = np.array(precisions)
recalls    = np.array(recalls)
f1s        = np.array(f1s)
fn_counts  = np.array(fn_counts)
fp_counts  = np.array(fp_counts)

# ── F1-optimal threshold ──────────────────────────────────────────────────────
best_f1_idx  = np.argmax(f1s)
THRESHOLD_F1 = thresholds[best_f1_idx]

print(f"F1-optimal threshold: {THRESHOLD_F1:.4f}")
print(f"  F1        at F1-opt : {f1s[best_f1_idx]:.4f}")
print(f"  Precision at F1-opt : {precisions[best_f1_idx]:.4f}")
print(f"  Recall    at F1-opt : {recalls[best_f1_idx]:.4f}")

# ── Plot: P / R / F1 vs threshold ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, precisions, color="#1E88E5", linewidth=2.0, label="Precision")
ax.plot(thresholds, recalls,    color="#F44336", linewidth=2.0, label="Recall")
ax.plot(thresholds, f1s,        color="#43A047", linewidth=2.5, label="F1 Score")

ax.axvline(x=THRESHOLD_F1, color="#43A047", linestyle="--", linewidth=1.5,
           label=f"F1-optimal  ({THRESHOLD_F1:.3f})")
ax.axvline(x=0.5, color="grey", linestyle=":", linewidth=1.5,
           label="Default  (0.5)")

ax.set_xlabel("Decision Threshold", fontsize=12)
ax.set_ylabel("Score", fontsize=12)
ax.set_title(
    f"Precision / Recall / F1 vs Threshold  [{BEST_MODEL_NAME}]",
    fontsize=13, fontweight="bold"
)
ax.legend(fontsize=10)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])
plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "threshold_sweep_prf1.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print(f" Figure saved → {FIGURES_DIR}/threshold_sweep_prf1.png")

In [ ]:
total_costs = FN_COST * fn_counts + FP_COST * fp_counts

best_cost_idx  = np.argmin(total_costs)
THRESHOLD_COST = thresholds[best_cost_idx]
MIN_COST       = total_costs[best_cost_idx]

# Metrics at default 0.5 for comparison
idx_05        = np.argmin(np.abs(thresholds - 0.5))
COST_DEFAULT  = total_costs[idx_05]
FN_AT_DEFAULT = fn_counts[idx_05]
FP_AT_DEFAULT = fp_counts[idx_05]

# Metrics at cost-optimal threshold
FN_AT_COST = fn_counts[best_cost_idx]
FP_AT_COST = fp_counts[best_cost_idx]

print("Business cost analysis:")
print(f"  FN cost : ${FN_COST:,} per missed fraud")
print(f"  FP cost : ${FP_COST:,}  per false alarm")
print()
print(f"  Default threshold (0.5):")
print(f"    FN missed   = {FN_AT_DEFAULT:,}   →  FN cost = ${FN_AT_DEFAULT * FN_COST:>10,.0f}")
print(f"    FP flagged  = {FP_AT_DEFAULT:,} →  FP cost = ${FP_AT_DEFAULT * FP_COST:>10,.0f}")
print(f"    Total cost  = ${COST_DEFAULT:,.0f}")
print()
print(f"  Cost-optimal threshold ({THRESHOLD_COST:.4f}):")
print(f"    FN missed   = {FN_AT_COST:,}   →  FN cost = ${FN_AT_COST * FN_COST:>10,.0f}")
print(f"    FP flagged  = {FP_AT_COST:,} →  FP cost = ${FP_AT_COST * FP_COST:>10,.0f}")
print(f"    Total cost  = ${MIN_COST:,.0f}")
print()
saving = COST_DEFAULT - MIN_COST
print(f"  Cost saving vs default threshold: ${saving:,.0f}")

# ── Plot: total cost vs threshold ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, total_costs, color="#FF5722", linewidth=2.5,
        label=f"Total cost  (${FN_COST}×FN + ${FP_COST}×FP)")
ax.axvline(x=THRESHOLD_COST, color="#E91E63", linestyle="--", linewidth=2,
           label=f"Cost-optimal  ({THRESHOLD_COST:.3f})  →  ${MIN_COST:,.0f}")
ax.axvline(x=0.5, color="grey", linestyle=":", linewidth=1.5,
           label=f"Default (0.5)  →  ${COST_DEFAULT:,.0f}")
ax.axvline(x=THRESHOLD_F1, color="#43A047", linestyle="-.", linewidth=1.5,
           label=f"F1-optimal  ({THRESHOLD_F1:.3f})")

ax.set_xlabel("Decision Threshold", fontsize=12)
ax.set_ylabel("Total Cost ($)", fontsize=12)
ax.set_title(
    f"Expected Cost vs Threshold  [{BEST_MODEL_NAME}]\n"
    f"FN cost = ${FN_COST} · FP cost = ${FP_COST}",
    fontsize=13, fontweight="bold"
)
ax.legend(fontsize=10)
plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "threshold_cost_curve.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print(f" Figure saved → {FIGURES_DIR}/threshold_cost_curve.png")

In [ ]:
DECISION_THRESHOLD = THRESHOLD_COST   # ← this is what the Streamlit app will use

y_pred_final = (BEST_MODEL_PROBS >= DECISION_THRESHOLD).astype(int)
tn_f, fp_f, fn_f, tp_f = confusion_matrix(y_test, y_pred_final, labels=[0, 1]).ravel()

final_recall    = tp_f / (tp_f + fn_f + 1e-9)
final_precision = tp_f / (tp_f + fp_f + 1e-9)
final_f1        = f1_score(y_test, y_pred_final, zero_division=0)

print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(f"  Final operating threshold: {DECISION_THRESHOLD:.4f}  (cost-optimal)")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print()
print(f"  Recall    : {final_recall:.4f}   ({final_recall*100:.1f}% of real fraud caught)")
print(f"  Precision : {final_precision:.4f}   ({final_precision*100:.1f}% of flags are real fraud)")
print(f"  F1        : {final_f1:.4f}")
print()
print(f"  True Positives  (fraud caught)    : {tp_f:,}")
print(f"  False Negatives (fraud missed)    : {fn_f:,}")
print(f"  False Positives (false alarms)    : {fp_f:,}")
print(f"  True Negatives  (legit passed)    : {tn_f:,}")
print()
print(f"  For comparison:")
print(f"    F1-optimal threshold   : {THRESHOLD_F1:.4f}  →  F1 = {f1s[best_f1_idx]:.4f}")
print(f"    Cost-optimal threshold : {THRESHOLD_COST:.4f}  →  Total cost = ${MIN_COST:,.0f}")
print()
print(f"  Chosen: {DECISION_THRESHOLD:.4f}  (business cost-optimal)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (threshold, label) in zip(
    axes,
    [
        (0.5,
         f"Default Threshold (0.5)\nsklearn out-of-the-box default"),
        (DECISION_THRESHOLD,
         f"Tuned Threshold ({DECISION_THRESHOLD:.3f})\ncost-optimal  [${FN_COST}×FN + ${FP_COST}×FP]"),
    ]
):
    y_pred_t = (BEST_MODEL_PROBS >= threshold).astype(int)
    cm_t = confusion_matrix(y_test, y_pred_t, labels=[0, 1])
    ConfusionMatrixDisplay(cm_t, display_labels=["Legitimate", "Fraud"]).plot(
        ax=ax, colorbar=False, cmap="Blues"
    )
    tn_t, fp_t, fn_t, tp_t = cm_t.ravel()
    rec_t  = tp_t / (tp_t + fn_t + 1e-9)
    prec_t = tp_t / (tp_t + fp_t + 1e-9)
    f1_t   = f1_score(y_test, y_pred_t, zero_division=0)
    cost_t = FN_COST * fn_t + FP_COST * fp_t
    ax.set_title(
        f"{label}\n"
        f"Recall={rec_t:.3f}  Precision={prec_t:.3f}  F1={f1_t:.3f}  "
        f"Cost=${cost_t:,.0f}",
        fontsize=10, fontweight="bold"
    )

plt.suptitle(
    f"Threshold Tuning Impact — {BEST_MODEL_NAME}",
    fontsize=13, fontweight="bold", y=1.02
)
plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "confusion_matrix_threshold_comparison.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print(f" Figure saved → {FIGURES_DIR}/confusion_matrix_threshold_comparison.png")

# ── Quantify the improvement ─────────────────────────────────────────────────
print()
print(f"  Default threshold (0.5)  :  {FN_AT_DEFAULT:,} missed frauds  →  cost ${COST_DEFAULT:,.0f}")
print(f"  Tuned threshold  ({DECISION_THRESHOLD:.3f}):  {FN_AT_COST:,} missed frauds  →  cost ${MIN_COST:,.0f}")
print()
print(f"  Additional fraud caught  : {FN_AT_DEFAULT - FN_AT_COST:,} transactions")
print(f"  Additional false alarms  : {FP_AT_COST - FP_AT_DEFAULT:,} transactions")
print(f"  Net cost saving          : ${COST_DEFAULT - MIN_COST:,.0f}")

In [ ]:
meta = joblib.load(MODEL_META)   # reload to avoid overwriting any concurrent edits

# ── Stage 9: evaluation results ──────────────────────────────────────────────
meta["TEST_PR_AUC"]  = {name: round(v, 6) for name, v in pr_auc_results.items()}
meta["TEST_ROC_AUC"] = {name: round(v, 6) for name, v in roc_auc_results.items()}

# ── Stage 10: threshold results ───────────────────────────────────────────────
meta["BEST_MODEL_NAME"]              = BEST_MODEL_NAME
meta["DECISION_THRESHOLD"]           = round(float(DECISION_THRESHOLD), 6)
meta["THRESHOLD_F1_OPT"]             = round(float(THRESHOLD_F1), 6)
meta["THRESHOLD_COST_OPT"]           = round(float(THRESHOLD_COST), 6)
meta["THRESHOLD_RATIONALE"]          = (
    f"Cost-optimal: minimises FN_COST×FN + FP_COST×FP "
    f"(FN=${FN_COST}, FP=${FP_COST}) on the held-out test set"
)
meta["TEST_RECALL_AT_THRESHOLD"]     = round(float(final_recall), 6)
meta["TEST_PRECISION_AT_THRESHOLD"]  = round(float(final_precision), 6)
meta["TEST_F1_AT_THRESHOLD"]         = round(float(final_f1), 6)
meta["FN_COST"]                      = FN_COST
meta["FP_COST"]                      = FP_COST

joblib.dump(meta, MODEL_META)

print(f" model_meta.joblib updated → {MODEL_META}")
print()
print("  Contents (all keys):")
for k, v in meta.items():
    v_str = str(v)
    display_v = v_str[:75] + ("..." if len(v_str) > 75 else "")
    print(f"    {k:<38}: {display_v}")

In [ ]:
print("=" * 62)
print("  STAGE 10 COMPLETE — THRESHOLD TUNING SUMMARY")
print("=" * 62)
print()
print(f"  Best model              : {BEST_MODEL_NAME}")
print(f"  PR-AUC (test)           : {pr_auc_results[BEST_MODEL_NAME]:.4f}")
print(f"  ROC-AUC (test)          : {roc_auc_results[BEST_MODEL_NAME]:.4f}")
print()
print(f"  F1-optimal threshold    : {THRESHOLD_F1:.4f}")
print(f"  Cost-optimal threshold  : {THRESHOLD_COST:.4f}  ← CHOSEN")
print()
print(f"  At chosen threshold ({DECISION_THRESHOLD:.4f}):")
print(f"    Recall    : {final_recall:.4f}   ({final_recall*100:.1f}% of fraud caught)")
print(f"    Precision : {final_precision:.4f}")
print(f"    F1        : {final_f1:.4f}")
print()
print(f"  Cost saving vs default 0.5 threshold: ${COST_DEFAULT - MIN_COST:,.0f}")
print()
print("  Figures saved to reports/figures/:")
for fname in [
    "pr_curves_comparison.png",
    "roc_curves_comparison.png",
    "confusion_matrices_default_threshold.png",
    "threshold_sweep_prf1.png",
    "threshold_cost_curve.png",
    "confusion_matrix_threshold_comparison.png",
]:
    print(f"    {fname}")
print()
print("  model_meta.joblib updated with:")
print("    BEST_MODEL_NAME, DECISION_THRESHOLD, THRESHOLD_RATIONALE,")
print("    TEST_PR_AUC, TEST_ROC_AUC,")
print("    TEST_RECALL/PRECISION/F1_AT_THRESHOLD,")
print("    FN_COST, FP_COST")
print()
print("  NEXT: Stage 11 — Feature Importance and SHAP Explainability")
print("  (understand exactly WHY the model flags specific transactions)")
print("=" * 62)